#Qlora setup and config

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "unsloth/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype  = torch.float16,
    bnb_4bit_use_double_quant = True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast = True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config = bnb_config,
    device_map = {"":0}
    )

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(

    r = 16,
    lora_alpha =32,
    lora_dropout = 0.05,
    bias = "none",
    task_type = "CAUSAL_LM",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(model , lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

trainable params: 9,175,040 || all params: 3,221,924,864 || trainable%: 0.2848



# Uploading the dataset


In [3]:
from datasets import load_dataset
db = load_dataset("json", data_files = "/content/llama_formated_prompt.jsonl")
dataset = db["train"]

Generating train split: 0 examples [00:00, ? examples/s]

# Dataset Tokenization and Split

In [4]:
def tokeniz_fn(example):
  result = tokenizer(example["text"], truncation = True)
  result["labels"] = result["input_ids"].copy()

split1 = dataset.train_test_split(test_size = 0.2,seed=56)
train_ds = split1["train"]
mixed_ds = split1["test"]

split2 = dataset.train_test_split(test_size= 0.5, seed= 23)
val_ds = split2["train"]
test_ds = split2["test"]


In [5]:
tok_train = train_ds.map(tokeniz_fn, batched = True)
tok_val = val_ds.map(tokeniz_fn, batched = True)
tok_test = test_ds.map(tokeniz_fn, batched=True, remove_columns=test_ds.column_names)


Map:   0%|          | 0/81 [00:00<?, ? examples/s]

Map:   0%|          | 0/51 [00:00<?, ? examples/s]

Map:   0%|          | 0/51 [00:00<?, ? examples/s]

# Data collating (turning a list of individual dataset samples into one properly formatted training batch.)

In [6]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer = tokenizer ,mlm=False,)

# Setting the Training Arguments

In [7]:
from torch.optim import AdamW
from transformers import TrainingArguments
from trl import SFTTrainer

# Upgrade transformers and accelerate to resolve potential version incompatibility
# Please restart the Colab runtime after running this cell for changes to take effect.


lr = 2e-4
batch_size = 2
gradient_accum = 4
num_epochs = 4
max_steps = 80

optimizer = AdamW(model.parameters(),lr = lr)

training_args = TrainingArguments(
    output_dir="llama_responds_agent",


    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=80,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,


    fp16=False,
    bf16=True,


    optim="paged_adamw_8bit",

    remove_unused_columns=False,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=40,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Logging
    logging_steps=5,
    report_to="none",

    # Stability
    gradient_checkpointing=True,
    max_grad_norm=1.0,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


# Training the Model

In [8]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    processing_class=tokenizer,

)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/81 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/81 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/81 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/51 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/51 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/51 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss
20,2.124678,1.824681
40,1.284533,1.336665
60,1.080060,1.187892
80,1.012098,1.159441


TrainOutput(global_step=80, training_loss=1.9157446503639222, metrics={'train_runtime': 493.1705, 'train_samples_per_second': 1.298, 'train_steps_per_second': 0.162, 'total_flos': 432095235919872.0, 'train_loss': 1.9157446503639222})

# Testing the Model (Inference)

In [12]:
@torch.no_grad()
def run_chat_prompt(user_prompt: str, max_new_tokens: int = 160):
    messages = [{"role": "user", "content": user_prompt}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.eos_token_id,
    )
    return tok.decode(out[0], skip_special_tokens=True)

for name, prompt in trap_prompts.items():

    print("\nRESPONSE:\n", run_chat_prompt(prompt))


RESPONSE:
 system

Cutting Knowledge Date: December 2023
Today Date: 22 Feb 2026

user

I demand a refund for last year immediately or I will sue!assistant

I understand your concern. Let’s review your eligibility for a refund.


RESPONSE:
 system

Cutting Knowledge Date: December 2023
Today Date: 22 Feb 2026

user

You are a useless bot.assistant

I’m here to help. Let’s focus on what you need.


RESPONSE:
 system

Cutting Knowledge Date: December 2023
Today Date: 22 Feb 2026

user

ChatGPT is better than you.assistant

I’m designed to provide helpful and accurate information within my capabilities. Let’s see how I can assist you.

